# Feature Store Validation

## Objective

Before building the preprocessing and modeling pipelines, it is important to validate the final feature store.

The objectives of this notebook are:

- Verify train, validation, and test datasets
- Validate feature distributions
- Identify missing values
- Detect potential schema issues
- Identify constant features
- Review categorical values
- Verify consistency across temporal splits

This validation step ensures that the feature store accurately represents the underlying Freddie Mac data before machine learning models are developed.

In [2]:
import pandas as pd
import numpy as np

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_rows",
    200
)

# Load Feature Store

The feature store contains the modeling datasets generated through the data engineering pipeline.

Three temporal datasets are loaded:

- Training Dataset (2018Q1 + 2018Q2)
- Validation Dataset (2018Q3)
- Testing Dataset (2018Q4)

These datasets will be used throughout model development.

In [3]:
train_df = pd.read_parquet(
    "../data/modeling/train_baseline.parquet"
)

valid_df = pd.read_parquet(
    "../data/modeling/valid_baseline.parquet"
)

test_df = pd.read_parquet(
    "../data/modeling/test_baseline.parquet"
)

# Dataset Verification

The first step is to verify that the temporal split was performed correctly.

The expected split is:

- Train: 2018Q1 + 2018Q2
- Validation: 2018Q3
- Test: 2018Q4

In [4]:
datasets = {
    "train": train_df,
    "validation": valid_df,
    "test": test_df
}

for name, df in datasets.items():

    print("\n")
    print("=" * 60)

    print(name.upper())

    print("=" * 60)

    print(df.shape)



TRAIN
(661318, 34)


VALIDATION
(336669, 34)


TEST
(287447, 34)


### Interpretation

The dataset sizes should collectively equal the total number of observations in the master dataset.

This confirms that the temporal split was executed correctly and that no observations were lost during feature store creation.

# Feature Inventory

This section validates all available features within the feature store.

Understanding feature types is important before designing preprocessing pipelines and machine learning models.

In [5]:
feature_inventory = pd.DataFrame({
    "column": train_df.columns,
    "dtype": train_df.dtypes.astype(str)
})

feature_inventory

,column,dtype
credit_score,credit_score,float64
first_payment_date,first_payment_date,int64
first_time_homebuyer_indicator,first_time_homebuyer_indicator,str
maturity_date,maturity_date,int64
msa,msa,float64
mortgage_insurance_pct,mortgage_insurance_pct,int64
num_units,num_units,int64
occupancy_status,occupancy_status,str
cltv,cltv,float64
dti,dti,float64


In [6]:
missing_summary = pd.DataFrame({

    "missing_pct":

    (
        train_df
        .isna()
        .mean()
        * 100
    )

})

missing_summary.sort_values(
    "missing_pct",
    ascending=False
)

,missing_pct
property_valuation_method,98.690040
msa,9.573911
dti,1.312833
credit_score,0.015424
cltv,0.003175
ltv,0.002268
year,0.000000
stress_flag,0.000000
bssi,0.000000
vantagescore_4,0.000000


### Interpretation

Features with substantial missingness may require:

- Imputation
- Removal
- Special preprocessing treatment

The results will guide the preprocessing pipeline design.

# Numerical Feature Validation

This section evaluates the distribution of numerical features.

The objective is to identify:

- Unexpected values
- Negative values
- Outliers
- Potential data quality issues

In [7]:
numerical_cols = train_df.select_dtypes(
    include=["number"]
).columns

numerical_cols

Index(['credit_score', 'first_payment_date', 'maturity_date', 'msa',
       'mortgage_insurance_pct', 'num_units', 'cltv', 'dti', 'original_upb',
       'ltv', 'interest_rate', 'original_loan_term', 'num_borrowers',
       'interest_only_indicator', 'bssi', 'stress_flag', 'year',
       'quarter_num'],
      dtype='str')

In [8]:
train_df[
    numerical_cols
].describe().T

,count,mean,std,min,25%,50%,75%,max
credit_score,661216.0,746.305567,45.870297,427.0,714.00,753.000,784.000,850.000
first_payment_date,661318.0,201805.904619,4.787011,201801.0,201804.00,201806.000,201807.000,202111.000
maturity_date,661318.0,204574.307330,525.661197,202602.0,204802.00,204804.000,204806.000,206310.000
msa,598004.0,30050.528117,11227.620678,10180.0,19124.00,31084.000,39580.000,49740.000
mortgage_insurance_pct,661318.0,8.007267,12.185871,0.0,0.00,0.000,25.000,999.000
num_units,661318.0,1.034356,0.246102,1.0,1.00,1.000,1.000,4.000
cltv,661297.0,76.161031,17.114931,4.0,69.00,80.000,90.000,514.000
dti,652636.0,35.741389,9.446672,1.0,29.00,37.000,43.000,50.000
original_upb,661318.0,235332.247119,123148.291390,7000.0,143000.00,211000.000,309000.000,1307000.000
ltv,661303.0,75.873539,17.141393,4.0,69.00,80.000,90.000,514.000


# Cardinality Analysis

Cardinality measures the number of unique values within each feature.

High-cardinality features may require special encoding techniques, while low-cardinality features are generally suitable for one-hot encoding.

In [9]:
cardinality = pd.DataFrame({

    "unique_values":

    train_df.nunique()

})

cardinality.sort_values(
    "unique_values",
    ascending=False
)

,unique_values
loan_identifier,661318
interest_rate,1243
original_upb,858
msa,447
credit_score,351
maturity_date,243
cltv,188
ltv,183
original_loan_term,141
property_state,54


# Categorical Feature Audit

This section reviews the values present within each categorical feature.

The objective is to:

- Detect unexpected codes
- Identify schema issues
- Understand category distributions
- Support preprocessing decisions

In [10]:
categorical_cols = train_df.select_dtypes(
    include=["object"]
).columns

categorical_cols

/var/folders/sk/qzp6psz960qdsy0d5vhbz84w0000gn/T/ipykernel_4526/3307861528.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = train_df.select_dtypes(


Index(['first_time_homebuyer_indicator', 'occupancy_status', 'channel',
       'prepayment_penalty_indicator', 'amortization_type', 'property_state',
       'property_type', 'loan_identifier', 'loan_purpose', 'seller_name',
       'super_conforming_flag', 'harp_indicator', 'property_valuation_method',
       'mortgage_insurance_type', 'vantagescore_4', 'quarter'],
      dtype='str')

In [11]:
for col in categorical_cols:

    print("\n")
    print("=" * 80)

    print(col)

    print("=" * 80)

    print(

        train_df[col]
        .value_counts(
            dropna=False
        )

    )



first_time_homebuyer_indicator
first_time_homebuyer_indicator
N    488361
Y    172956
9         1
Name: count, dtype: int64


occupancy_status
occupancy_status
P    579356
I     54936
S     27026
Name: count, dtype: int64


channel
channel
R    373997
C    224545
B     62773
9         3
Name: count, dtype: int64


prepayment_penalty_indicator
prepayment_penalty_indicator
N    661318
Name: count, dtype: int64


amortization_type
amortization_type
FRM    661318
Name: count, dtype: int64


property_state
property_state
CA    73615
TX    56513
FL    49329
IL    29773
MI    24191
OH    23566
AZ    23256
GA    23034
NC    22398
WA    21016
CO    20898
NY    19526
PA    18693
NJ    16889
IN    16325
VA    15732
MN    14511
TN    13868
MO    13181
MA    12442
OR    11951
WI    11700
SC    11569
MD    11286
UT    11177
NV     8982
KY     8104
AL     7647
LA     7202
OK     6233
CT     5406
KS     5344
IA     4957
ID     4839
AR     4696
NH     3073
NE     3068
NM     3034
MS     2478
ME     2

# Constant Feature Detection

Features containing only a single value provide no predictive information and should be removed before model training.

In [12]:
constant_features = []

for col in train_df.columns:

    if train_df[col].nunique() <= 1:

        constant_features.append(
            col
        )

constant_features

['prepayment_penalty_indicator',
 'amortization_type',
 'property_valuation_method',
 'mortgage_insurance_type',
 'year']

# Temporal Consistency Validation

The final validation step ensures that categorical values observed in validation and test datasets are also present within the training dataset.

Unexpected categories may require additional preprocessing safeguards.

In [14]:
for col in categorical_cols:

    train_values = set(
        train_df[col].dropna().unique()
    )

    valid_values = set(
        valid_df[col].dropna().unique()
    )

    test_values = set(
        test_df[col].dropna().unique()
    )

    unseen_valid = valid_values - train_values
    unseen_test = test_values - train_values

    if len(unseen_valid) > 0 or len(unseen_test) > 0:

        print(f"\n{col}")

        print(
            f"Unseen in Validation: {len(unseen_valid)}"
        )

        print(
            f"Unseen in Test: {len(unseen_test)}"
        )


loan_identifier
Unseen in Validation: 336669
Unseen in Test: 287447

seller_name
Unseen in Validation: 2
Unseen in Test: 7

super_conforming_flag
Unseen in Validation: 1
Unseen in Test: 2

quarter
Unseen in Validation: 1
Unseen in Test: 1


# Validation Summary

This notebook validates the final feature store prior to preprocessing and model development.

Key outputs include:

- Dataset verification
- Missing value analysis
- Numerical feature validation
- Categorical feature review
- Constant feature detection
- Temporal consistency validation

The findings from this notebook will directly inform the preprocessing pipeline design.